# 04 End-to-End Inference and Visual Grounding

This notebook demonstrates loading the model, generating a structured report, computing Grad-CAM overlays, compiling evidence logs, and running post-generation factuality checks.

In [ ]:
import os
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))
from src.config import get_config
from src.vision_encoder import GroundedVisionEncoder
from src.grounding_module import GradCAMGrounding
from src.report_generator import ConstrainedReportGenerator
from src.hallucination_detector import HallucinationDetector

In [ ]:
config = get_config()
# Load encoder
model = GroundedVisionEncoder(config=config.vision)
model.eval()

# Load grounding and generator
grounding = GradCAMGrounding(model, config=config.gradcam)
generator = ConstrainedReportGenerator(model, grounding, config=config.report)

In [ ]:
# Generate mock chest X-ray image for testing
dummy_arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
image = Image.fromarray(dummy_arr)

# Run generator pipeline
output = generator.generate(
    image=image,
    patient_history="Patient has shortness of breath."
)

print("=== GENERATED REPORT ===")
print(output.report_text)

In [ ]:
print("=== EVIDENCE LOG ===")
evidence_df = pd.DataFrame(output.evidence_log)
display(evidence_df)

In [ ]:
print("=== POST-GENERATION VERIFICATION ===")
for check in output.hallucination_flags:
    print(f"Claim: {check['claim']}")
    print(f"  Hallucinated: {check['hallucinated']}")
    print(f"  Explanation: {check['explanation']}")